# Experiment 6: Model Evaluation

**Dataset:** Heart Disease Cleveland Dataset

### Objective
Evaluate the trained Logistic Regression, Random Forest, and SVM models on the untouched test dataset using accuracy, precision, recall, F1-score, ROC-AUC, confusion matrices, ROC curves, and per-class recall.

## Step 1: Import Required Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)


## Step 2: Load the Dataset

We load the original dataset and recreate the same stratified 80/20 split used in Experiment 5. The test set is not used for training.

In [ ]:
df = pd.read_csv("Heart_disease_cleveland_new.csv")

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Test data:", X_test.shape)


## Step 3: Load the Trained Models

These are the complete pipelines saved in Experiment 5. Each pipeline contains preprocessing and the trained classifier.

In [ ]:
models = {
    "Logistic Regression": joblib.load("logistic_regression.pkl"),
    "Random Forest": joblib.load("random_forest.pkl"),
    "SVM": joblib.load("svm.pkl")
}

print("Models loaded successfully.")


## Step 4: Calculate Evaluation Metrics

We evaluate all three models on the held-out test set. Recall is given particular importance because the project aims to identify patients with heart disease.

In [ ]:
results = []
predictions = {}
probabilities = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    predictions[name] = y_pred
    probabilities[name] = y_prob
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)
results_df.round(3)


## Step 5: Compare the Models

This table gives the main test-set performance of the three models.

In [ ]:
results_df.round(3)


## Step 6: Confusion Matrix

A confusion matrix shows:
- True Negative (TN)
- False Positive (FP)
- False Negative (FN)
- True Positive (TP)

In [ ]:
for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["No Disease", "Disease"],
        yticklabels=["No Disease", "Disease"]
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - {name}")
    plt.show()


## Step 7: Classification Report and Per-Class Recall

The classification report provides precision, recall, and F1-score for both classes. This helps us see whether the model performs differently for disease and no-disease cases.

In [ ]:
for name, y_pred in predictions.items():
    print("=" * 60)
    print(name)
    print("=" * 60)
    print(classification_report(
        y_test,
        y_pred,
        target_names=["No Disease (0)", "Disease (1)"],
        zero_division=0
    ))


## Step 8: Plot ROC Curves

The ROC curve shows the relationship between True Positive Rate and False Positive Rate at different classification thresholds. ROC-AUC summarizes the model's ability to distinguish the two classes.

In [ ]:
plt.figure(figsize=(7, 5))

for name, y_prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.show()


## Step 9: Select the Final Model

For this project, recall for the disease-positive class is the primary metric because missing a disease-positive case is important in a clinical decision-support setting. F1-score is used as the tie-breaker when recall values are equal.

In [ ]:
final_model_row = results_df.sort_values(
    by=["Recall", "F1-Score"],
    ascending=False
).iloc[0]

final_model_name = final_model_row["Model"]

print("Selected final model:", final_model_name)
print("Reason: It has the highest test-set recall; F1-score is used as the tie-breaker.")


## Step 10: Final Evaluation Summary

After running the notebook, record the actual values from the results table in the lab report. Do not manually invent or estimate the scores.

### Conclusion
The three trained models were evaluated using the held-out test dataset. Accuracy, precision, recall, F1-score, ROC-AUC, confusion matrices, ROC curves, and per-class recall were examined. The final model was selected using disease-positive recall as the primary criterion and F1-score as the tie-breaker. The selected model can be used as the model artifact for Experiment 7 deployment.